# ML Basics & Linear Regression — Boston Housing Assignment

**Target:** `medv` (median house value).  
**Dataset:** `BostonHousing.csv` supplied with the assignment.  
This notebook answers the theoretical questions and completes the practical workflow. Run the cells from top to bottom in Google Colab. Upload `BostonHousing.csv` to Colab when prompted, or keep it in the same folder as this notebook.

## Part A — Theoretical Questions

### Q1. What is Machine Learning? Explain the main types with examples.
Machine Learning (ML) is a branch of artificial intelligence in which systems learn patterns from data to make predictions or decisions without being explicitly programmed for every case.

- **Supervised learning:** learns from labelled examples. Examples: predicting house prices (regression), detecting spam (classification).
- **Unsupervised learning:** finds structure in unlabelled data. Examples: customer segmentation using clustering, grouping similar documents.
- **Reinforcement learning:** an agent learns by taking actions and receiving rewards or penalties. Examples: game-playing agents and robot navigation.

### Q2. What is Supervised Learning? Difference between Regression and Classification.
Supervised learning trains a model using input features paired with known target labels.

- **Regression** predicts a continuous numeric value, such as a house price or temperature.
- **Classification** predicts a discrete class/category, such as spam/not spam or disease/no disease.

### Q3. What is Linear Regression? Equation, intercept and coefficient.
Linear Regression models a numeric target as a linear combination of input features:

\[
\hat{y}=\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_px_p
\]

Here, \(\hat{y}\) is the predicted target, \(\beta_0\) is the **intercept** (predicted target when all features equal zero), and \(\beta_i\) is the **coefficient** for feature \(x_i\). A coefficient describes the expected change in prediction for a one-unit increase in that feature, holding the other features constant.

### Q4. Explain SSE, MSE, RMSE, R² and Adjusted R².
- **SSE (Sum of Squared Errors):** \(\sum_i(y_i-\hat{y}_i)^2\); total squared prediction error. Lower is better.
- **MSE (Mean Squared Error):** \(\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2\); average squared error. Lower is better.
- **RMSE (Root Mean Squared Error):** \(\sqrt{MSE}\); error measured in the target's original units. Lower is better.
- **R² score:** \(1-\frac{SSE}{SST}\); proportion of target variance explained by the model. A value closer to 1 generally indicates a better fit; it can be negative on test data.
- **Adjusted R²:** adjusts R² for the number of predictors and sample size. It can decrease when unhelpful predictors are added. On a test set, this notebook uses \(1-(1-R^2)\frac{n-1}{n-p-1}\), where \(n\) is the number of test observations and \(p\) is the number of predictors.

### Q5. Machine Learning workflow from data collection to evaluation.
1. Define the problem and target.
2. Collect and load data.
3. Inspect, clean, and prepare data (types, missing values, duplicates, outliers as appropriate).
4. Explore data using descriptive statistics and visualizations.
5. Select features and target.
6. Split data into training and testing sets.
7. Train a suitable model using training data.
8. Predict on unseen test data.
9. Evaluate using suitable metrics and inspect errors.
10. Interpret results, improve the model if needed, and communicate findings.

## Part B — Practical Questions (Q6–Q10)

In [ ]:
# If running in Google Colab, upload BostonHousing.csv when prompted.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from google.colab import files
    if not os.path.exists("BostonHousing.csv"):
        uploaded = files.upload()
except ImportError:
    pass

df = pd.read_csv("BostonHousing.csv")
print("Dataset shape:", df.shape)
display(df.head())

### Q6. Basic data exploration: `head()`, `info()`, `describe()`

In [ ]:
# First five rows
display(df.head())

# Column names, non-null counts, data types, and memory usage
df.info()

# Descriptive statistics for numeric columns
display(df.describe())

### Data quality check

In [ ]:
print("Missing values per column:")
display(df.isnull().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
print("Columns:", df.columns.tolist())

### Q7. EDA: pairplot, correlation heatmap, and features related to `medv`

The pairplot focuses on `medv` and four useful features so the chart remains readable. The heatmap uses all numeric columns.

In [ ]:
# Pairplot of target and selected features
pairplot_cols = [c for c in ["medv", "rm", "lstat", "ptratio", "crim"] if c in df.columns]
sns.pairplot(df[pairplot_cols], diag_kind="hist", corner=True)
plt.suptitle("Pairplot: Boston Housing Selected Features", y=1.02)
plt.show()

# Correlation heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Boston Housing")
plt.tight_layout()
plt.show()

# Correlation of each feature with medv
medv_corr = df.corr(numeric_only=True)["medv"].drop("medv").sort_values(ascending=False)
display(medv_corr.to_frame("correlation_with_medv"))

print("Strongest positive correlation:", medv_corr.idxmax(), round(medv_corr.max(), 3))
print("Strongest negative correlation:", medv_corr.idxmin(), round(medv_corr.min(), 3))

**Interpretation:** In this dataset, `rm` (average number of rooms) has the strongest positive linear correlation with `medv`, while `lstat` (lower-status population measure) has the strongest negative correlation. Correlation is an association, not proof of causation.

### Q8. Select independent variables and dependent variable

The target is `medv`. All other columns are used as independent variables.

In [ ]:
X = df.drop(columns=["medv"])
y = df["medv"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", X.columns.tolist())

### Q9. Split data into 80% training and 20% testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

### Q10. Train Linear Regression, predict, evaluate, write equation, interpret coefficients

In [ ]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
sse = np.sum((y_test - y_pred) ** 2)
r2 = r2_score(y_test, y_pred)

n = len(y_test)       # number of test observations
p = X_test.shape[1]   # number of predictors
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

metrics = pd.DataFrame({
    "Metric": ["SSE", "MAE", "MSE", "RMSE", "R² Score", "Adjusted R²"],
    "Value": [sse, mae, mse, rmse, r2, adjusted_r2]
})
display(metrics.round(4))

In [ ]:
# Linear regression equation
print(f"medv = {model.intercept_:.4f}", end="")
for feature, coef in zip(X.columns, model.coef_):
    sign = "+" if coef >= 0 else "-"
    print(f" {sign} {abs(coef):.4f}*{feature}", end="")
print()

# Coefficient table
coef_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})
coef_table["Direction"] = np.where(coef_table["Coefficient"] >= 0, "Positive", "Negative")
display(coef_table.sort_values("Coefficient", key=lambda s: s.abs(), ascending=False).round(4))

print(f"Intercept: {model.intercept_:.4f}")
print("Interpretation: each coefficient is the predicted change in medv for a one-unit increase in that feature, holding all other features constant.")

### Example interpretation of the coefficients
- A **positive coefficient** means the model predicts higher `medv` as that feature increases, with other features held constant.
- A **negative coefficient** means the model predicts lower `medv` as that feature increases, with other features held constant.
- Coefficients are conditional on the other included predictors; correlated features can make individual coefficients less stable.
- `medv` is expressed in thousands of dollars in the classic Boston Housing dataset, so a prediction such as 25 corresponds to approximately $25,000 in the dataset's original scale.

### Final conclusion
The model is evaluated on a held-out 20% test set. Use the reported MAE/RMSE to understand typical prediction error and R² to understand the proportion of test-target variation explained. These results are a baseline; consider cross-validation and checking residuals before using a model for real decisions.